# Chapter 16: Hyperbolic Geometry

Source orientation: printed pages 287-306; PDF pages 305-324. I inspected the rendered source span privately for section structure and terminology: Euclidean and hyperbolic parallel axioms, relative consistency by models, conformal and projective disk models, angle of parallelism, asymptotic triangles, finite hyperbolic area and angular defect, circles/horocycles/equidistant curves, Poincare's half-plane model, and the horosphere as a Euclidean surface. This notebook is original teaching material and does not reproduce textbook prose, exercises, screenshots, page crops, or source figures.

## Chapter Goal

Hyperbolic geometry becomes plausible when the parallel axiom is visualized as a model choice rather than a philosophical threat. The chapter's central move is to keep the absolute-geometric background and replace one parallel assertion: instead of a unique parallel through a point, more than one line through that point may avoid a given line. From that one change flow conformal and projective models, limiting parallels, finite area for ideal triangles, angular defect, horocycles, equidistant curves, and the surprising fact that a horosphere carries Euclidean geometry.

This notebook is built around model comparison. The Poincare disk and upper half-plane keep angles but distort distance. The Klein disk keeps geodesics straight but distorts angles. The half-plane turns horocycles into horizontal lines and lets formulas such as `Pi(x) = 2 arctan(exp(-x))` become plots. The computational checks do not prove consistency in the formal logical sense, but they expose why a model can make the hyperbolic axiom coherent and why the resulting geometry is not merely Euclidean geometry drawn strangely.

The learner should keep asking three questions: Which curves are hyperbolic lines? Which quantities are model artifacts? Which invariants survive across the model change?

## Visualization Storyboard

| Section | Representation | Artifact | Inspection target | Check |
|---|---|---|---|---|
| Parallel axiom | Euclidean line panel and half-plane geodesic panel | `parallel-axiom-model-comparison.png` | unique Euclidean parallel versus many hyperbolic nonintersecting geodesics | intersection classification for semicircles |
| Conformal/projective models | Poincare disk arcs and Klein chords | `poincare-klein-model-comparison.png` | same ideal endpoints, different visual promises | disk-to-Klein map keeps points inside disk |
| Angle of parallelism | formula plot and limiting ray diagram | `angle-of-parallelism-function.png` | `Pi(x)` decreases from a right angle toward zero | monotonicity and sample values |
| Area and angular defect | angle budget diagrams and functional check | `angular-defect-area-budget.png` | area is proportional to `pi - A - B - C` | additivity table and nonnegative defect |
| Circles, horocycles, equidistant curves | upper half-plane pencils | `half-plane-pencils-horocycles-equidistant.png` | ordinary circles, horocycles, and equidistant curves are distinct families | distance-to-geodesic level check |
| Horosphere analogy | 3D upper-half-space sketch | `horosphere-euclidean-limit.html` | horizontal horospheres inherit Euclidean geometry | equal Euclidean displacements on a horosphere |

Matplotlib is used for inspectable 2D proof diagrams. Plotly is used for the 3D horosphere view because rotating the upper-half-space picture makes the limiting Euclidean plane idea much clearer. NumPy supplies the formulas, intersections, and inequality checks.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display

CHAPTER_NO = 16
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

CHAPTER = chapter_by_no(CHAPTER_NO)
DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = DIRS["figures"]
HTML = DIRS["html"]
CHECKS = DIRS["checks"]
TABLES = DIRS["tables"]

for stale in [FIG / "concept_configuration.svg", FIG / "parameter_experiment.svg", TABLES / "artifact_manifest.csv"]:
    if stale.exists():
        stale.unlink()

COLORS = {
    "ink": "#111827",
    "muted": "#6b7280",
    "blue": "#2563eb",
    "green": "#059669",
    "orange": "#d97706",
    "red": "#dc2626",
    "purple": "#7c3aed",
}

visual_artifacts: list[Path] = []
check_artifacts: list[Path] = []
table_artifacts: list[Path] = []
computed_checks: dict[str, object] = {}

def save_png(fig: plt.Figure, filename: str) -> Path:
    path = FIG / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    visual_artifacts.append(path)
    return path

def display_many(paths: list[Path], width: int = 760) -> None:
    for path in paths:
        display(Markdown(f"### {path.name}"))
        display_artifact(path, width=width)

def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def upper_halfplane_geodesic(center: float, through_y: float, samples: int = 260) -> tuple[np.ndarray, float]:
    radius = math.sqrt(center * center + through_y * through_y)
    theta = np.linspace(0.02, math.pi - 0.02, samples)
    z = center + radius * np.cos(theta) + 1j * radius * np.sin(theta)
    return z, radius

def circles_intersect(c1: float, r1: float, c2: float, r2: float) -> bool:
    d = abs(c1 - c2)
    return abs(r1 - r2) <= d <= (r1 + r2)


## 1. One Parallel Or Many?

In the Euclidean plane, a point outside a line has exactly one line through it that does not meet the original line. The hyperbolic axiom replaces that with more than one such line. The important word is "line": in a model, a hyperbolic line may be drawn as a Euclidean arc. If the model satisfies the incidence and order rules, the arc is not a fake line; it is the model's representation of one.

The upper-half-plane panel uses geodesics that are vertical rays or semicircles perpendicular to the boundary. The fixed line `r` is one such semicircle. Several geodesics through `A` do not meet it; two are limiting parallels, and farther ones cross it. This is the cleanest way to make the hyperbolic axiom visible without pretending that Euclidean straightness is the definition of linehood.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5.4))

ax = axs[0]
ax.axhline(0, color=COLORS["ink"], lw=2.0, label="line r")
A_e = np.array([0.0, 1.35])
ax.scatter([A_e[0]], [A_e[1]], s=70, color=COLORS["red"], zorder=4)
ax.text(A_e[0] + 0.05, A_e[1] + 0.08, "A", fontsize=11)
for slope in [-1.1, -0.45, 0.0, 0.55, 1.2]:
    xs = np.linspace(-2.2, 2.2, 100)
    ys = A_e[1] + slope * xs
    color = COLORS["green"] if abs(slope) < 1e-12 else COLORS["muted"]
    lw = 2.6 if abs(slope) < 1e-12 else 1.2
    ax.plot(xs, ys, color=color, lw=lw, alpha=0.85)
ax.set_title("Euclidean model: one nonmeeting line through A")
ax.set_xlim(-2.3, 2.3)
ax.set_ylim(-0.4, 2.45)
ax.grid(alpha=0.25)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")

ax = axs[1]
theta = np.linspace(0.02, math.pi - 0.02, 260)
r_line = np.exp(1j * theta)
ax.plot(r_line.real, r_line.imag, color=COLORS["ink"], lw=2.2, label="hyperbolic line r")
A_y = 2.0
ax.scatter([0], [A_y], s=70, color=COLORS["red"], zorder=4)
ax.text(0.05, A_y + 0.07, "A", fontsize=11)
classification_rows = []
for center in [-2.2, -1.5, -1.0, -0.4, 0.4, 1.0, 1.5, 2.2]:
    geo, radius = upper_halfplane_geodesic(center, A_y)
    meets = circles_intersect(center, radius, 0.0, 1.0)
    limiting = abs(center) == 1.5
    color = COLORS["orange"] if meets else COLORS["blue"]
    if limiting:
        color = COLORS["purple"]
    ax.plot(geo.real, geo.imag, color=color, lw=2.0 if limiting else 1.25, alpha=0.82)
    classification_rows.append({
        "circle_center": center,
        "circle_radius": radius,
        "meets_r": bool(meets and not limiting),
        "limiting_parallel": bool(limiting),
    })
ax.axhline(0, color=COLORS["muted"], lw=1.0)
ax.set_title("Hyperbolic half-plane: many nonmeeting geodesics")
ax.set_xlim(-3.2, 3.2)
ax.set_ylim(-0.05, 3.35)
ax.grid(alpha=0.25)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("boundary coordinate")
ax.set_ylabel("height")
handles = [
    plt.Line2D([0], [0], color=COLORS["blue"], lw=2, label="does not meet r"),
    plt.Line2D([0], [0], color=COLORS["purple"], lw=2, label="limiting parallel"),
    plt.Line2D([0], [0], color=COLORS["orange"], lw=2, label="meets r"),
]
ax.legend(handles=handles, loc="upper right")

parallel_path = save_png(fig, "parallel-axiom-model-comparison.png")
parallel_table = write_csv(TABLES / "half-plane-parallel-classification.csv", classification_rows)
table_artifacts.append(parallel_table)
computed_checks["parallel_axiom"] = {
    "sampled_geodesics": len(classification_rows),
    "nonmeeting_count": sum(not row["meets_r"] for row in classification_rows),
    "limiting_parallel_count": sum(row["limiting_parallel"] for row in classification_rows),
}
assert computed_checks["parallel_axiom"]["nonmeeting_count"] > 2
assert computed_checks["parallel_axiom"]["limiting_parallel_count"] == 2
display_many([parallel_path])


## 2. Conformal And Projective Disk Models

The source span compares two Euclidean representations of the same hyperbolic plane. In the Poincare disk, hyperbolic lines are arcs that meet the boundary circle at right angles; the model is conformal, so angles at ordinary points are represented honestly. In the Klein disk, hyperbolic lines are chords; the model is projective, so straightness is represented honestly, but angles are not.

The next figure uses the same ideal endpoints in both panels. That is the inspection target: a model can keep one feature visually simple by making another feature less literal. The code also records the standard map from Poincare disk coordinates to Klein disk coordinates, `k = 2p/(1 + |p|^2)`, and checks that sample points remain inside the disk.

In [ ]:
def orthogonal_arc(theta1: float, theta2: float, samples: int = 320) -> np.ndarray:
    u = np.array([math.cos(theta1), math.sin(theta1)])
    v = np.array([math.cos(theta2), math.sin(theta2)])
    center = np.linalg.solve(np.vstack([u, v]), np.array([1.0, 1.0]))
    radius = math.sqrt(float(center @ center) - 1.0)
    angles = np.linspace(0, 2 * math.pi, samples)
    pts = center[0] + radius * np.cos(angles) + 1j * (center[1] + radius * np.sin(angles))
    inside = np.abs(pts) <= 1.0001
    pts = pts[inside]
    # Keep the component whose midpoint angle sits between the endpoint directions.
    return pts

endpoint_pairs = [(-2.55, -0.65), (-2.05, 0.85), (-1.35, 1.95), (-0.45, 2.55)]
sample_points = np.array([0.1 + 0.2j, -0.25 + 0.35j, 0.45 - 0.1j, 0.0 + 0.65j])
klein_points = 2 * sample_points / (1 + np.abs(sample_points) ** 2)

fig, axs = plt.subplots(1, 2, figsize=(12, 5.8))
circle = np.exp(1j * np.linspace(0, 2 * np.pi, 500))
for ax, title in zip(axs, ["Poincare disk: conformal arcs", "Klein disk: straight chords"]):
    ax.plot(circle.real, circle.imag, color=COLORS["ink"], lw=1.6)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.08, 1.08)
    ax.set_ylim(-1.08, 1.08)
    ax.grid(alpha=0.25)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

for t1, t2 in endpoint_pairs:
    arc = orthogonal_arc(t1, t2)
    axs[0].plot(arc.real, arc.imag, color=COLORS["blue"], lw=2.0)
    p1, p2 = np.exp(1j * t1), np.exp(1j * t2)
    axs[1].plot([p1.real, p2.real], [p1.imag, p2.imag], color=COLORS["orange"], lw=2.0)
axs[0].scatter(sample_points.real, sample_points.imag, s=52, color=COLORS["red"], label="sample points")
axs[1].scatter(klein_points.real, klein_points.imag, s=52, color=COLORS["red"], label="mapped sample points")
axs[0].legend(loc="lower left")
axs[1].legend(loc="lower left")

models_path = save_png(fig, "poincare-klein-model-comparison.png")
computed_checks["models"] = {
    "sample_point_count": len(sample_points),
    "max_poincare_radius": float(np.abs(sample_points).max()),
    "max_klein_radius": float(np.abs(klein_points).max()),
}
assert np.all(np.abs(sample_points) < 1)
assert np.all(np.abs(klein_points) < 1)
display_many([models_path])


## 3. The Angle Of Parallelism

The angle of parallelism `Pi(x)` is the angle made by a limiting parallel when the perpendicular distance to the reference line is `x`. In the half-plane normalization used here, the formula is

`Pi(x) = 2 arctan(exp(-x))`.

This is one of the most useful computational handles in the chapter. It starts at a right angle when the point approaches the line, then decreases steadily toward zero as the point recedes. The plot records that monotonic behavior and the table gives concrete values a learner can check.

In [ ]:
x = np.linspace(0, 5.5, 400)
Pi = 2 * np.arctan(np.exp(-x))
sample_x = np.array([0, 0.5, 1, 2, 3, 5], dtype=float)
sample_Pi = 2 * np.arctan(np.exp(-sample_x))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.2))
ax = axs[0]
ax.plot(x, Pi, color=COLORS["blue"], lw=2.5)
ax.scatter(sample_x, sample_Pi, color=COLORS["red"], s=45)
ax.set_title("angle of parallelism decreases with distance")
ax.set_xlabel("distance x")
ax.set_ylabel("Pi(x), radians")
ax.set_ylim(0, math.pi / 2 + 0.1)
ax.grid(alpha=0.25)

ax = axs[1]
base = np.array([0.0, 0.0])
height = 1.7
for dist, color in [(0.4, COLORS["green"]), (1.2, COLORS["orange"]), (2.4, COLORS["purple"])]:
    angle = 2 * math.atan(math.exp(-dist))
    A = np.array([dist, height])
    ax.scatter([A[0]], [A[1]], color=color, s=55)
    ax.plot([A[0], A[0]], [0, A[1]], color=COLORS["muted"], lw=1.0, ls="--")
    for sign in [-1, 1]:
        direction = np.array([sign * math.cos(angle), -math.sin(angle)])
        end = A + 1.15 * direction
        ax.add_patch(FancyArrowPatch(A, end, arrowstyle="->", mutation_scale=11, lw=1.8, color=color))
    ax.text(A[0] + 0.04, A[1] + 0.06, f"x={dist}", fontsize=9)
ax.axhline(0, color=COLORS["ink"], lw=1.4)
ax.set_title("limiting directions narrow as A moves away")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.7, 3.4)
ax.set_ylim(-0.2, 2.2)
ax.grid(alpha=0.25)
ax.set_xlabel("model coordinate")
ax.set_ylabel("height")

angle_path = save_png(fig, "angle-of-parallelism-function.png")
angle_rows = [{"distance": float(a), "Pi_radians": float(b), "Pi_degrees": float(np.degrees(b))} for a, b in zip(sample_x, sample_Pi)]
angle_table = write_csv(TABLES / "angle-of-parallelism-values.csv", angle_rows)
table_artifacts.append(angle_table)
computed_checks["angle_of_parallelism"] = {
    "Pi_0": float(sample_Pi[0]),
    "Pi_5": float(sample_Pi[-1]),
    "strictly_decreasing": bool(np.all(np.diff(Pi) < 0)),
}
assert abs(sample_Pi[0] - math.pi / 2) < 1e-12
assert np.all(np.diff(Pi) < 0)
display_many([angle_path])


## 4. Finite Area And Angular Defect

Hyperbolic triangles have angle sum less than `pi`. The source span uses asymptotic triangles and Gauss's functional argument to reach the area law: with a suitable unit, the area of a finite triangle is `pi - A - B - C`. This is not an arbitrary formula. It says area is controlled by angular defect, the amount by which the angle sum falls short of two right angles.

The figure below compares several angle triples. The check table emphasizes additivity: cutting a hyperbolic triangle into smaller pieces preserves total defect, so the area budget behaves like a geometric measure. Spherical geometry is mentioned in the source as a formal analogy with the sign reversed; the notebook keeps the hyperbolic side central but includes the sign comparison so the learner can see why negative curvature is the right intuition.

In [ ]:
triangles = [
    ("nearly Euclidean", np.radians([62, 58, 55])),
    ("moderate defect", np.radians([50, 47, 42])),
    ("large defect", np.radians([35, 30, 25])),
    ("asymptotic-style", np.radians([55, 0, 35])),
]
area_rows = []
for name, angles in triangles:
    defect = math.pi - float(np.sum(angles))
    area_rows.append({
        "triangle": name,
        "A_degrees": float(np.degrees(angles[0])),
        "B_degrees": float(np.degrees(angles[1])),
        "C_degrees": float(np.degrees(angles[2])),
        "angle_sum_degrees": float(np.degrees(np.sum(angles))),
        "angular_defect": defect,
        "area_mu_1": defect,
    })

phi = np.linspace(0, math.pi, 200)
f_phi = phi
complement_sum = f_phi + (math.pi - phi)
psi = math.pi / 5
additive_error = np.max(np.abs((phi[:-60] + psi) - phi[:-60] - psi))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.1))
ax = axs[0]
labels = [row["triangle"] for row in area_rows]
defects = [row["angular_defect"] for row in area_rows]
angle_sums = [math.pi - d for d in defects]
xs = np.arange(len(labels))
ax.bar(xs - 0.18, angle_sums, width=0.36, color=COLORS["blue"], label="angle sum")
ax.bar(xs + 0.18, defects, width=0.36, color=COLORS["orange"], label="defect = area")
ax.axhline(math.pi, color=COLORS["ink"], lw=1.2, ls="--", label="pi")
ax.set_xticks(xs)
ax.set_xticklabels(labels, rotation=18, ha="right")
ax.set_ylabel("radians")
ax.set_title("hyperbolic area is angular defect")
ax.grid(axis="y", alpha=0.25)
ax.legend()

ax = axs[1]
ax.plot(phi, f_phi, color=COLORS["green"], lw=2.3, label="f(phi)=phi when treble area is pi")
ax.plot(phi, complement_sum, color=COLORS["purple"], lw=1.8, label="f(phi)+f(pi-phi)=pi")
ax.axhline(math.pi, color=COLORS["ink"], lw=1.0, ls="--")
ax.set_xlabel("phi")
ax.set_ylabel("functional value")
ax.set_title("Gauss-style functional scaffold")
ax.grid(alpha=0.25)
ax.legend()

defect_path = save_png(fig, "angular-defect-area-budget.png")
area_table = write_csv(TABLES / "angular-defect-area-values.csv", area_rows)
table_artifacts.append(area_table)
computed_checks["area_defect"] = {
    "triangle_count": len(area_rows),
    "all_defects_positive": all(row["angular_defect"] >= 0 for row in area_rows),
    "complement_identity_error": float(np.max(np.abs(complement_sum - math.pi))),
    "additivity_error": float(additive_error),
}
assert computed_checks["area_defect"]["all_defects_positive"]
assert computed_checks["area_defect"]["complement_identity_error"] < 1e-12
display_many([defect_path])


## 5. Circles, Horocycles, And Equidistant Curves

The source span distinguishes three pencils of lines and three orthogonal trajectory families. In the upper half-plane, this becomes very concrete. Hyperbolic lines are vertical rays or semicircles perpendicular to the boundary. Horocycles with an ideal center at infinity are horizontal Euclidean lines. Curves at constant distance from a vertical geodesic are Euclidean rays through the boundary point; they are not geodesics unless they are vertical.

The code checks the constant-distance claim using the formula `distance_to_vertical = asinh(|x|/y)`. Along a ray `x = k y`, this quantity is constant. This is a good example of model literacy: a Euclidean straight ray can represent a hyperbolic equidistant curve rather than a hyperbolic line.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.8))
ax.axhline(0, color=COLORS["ink"], lw=1.4, label="boundary")

# Hyperbolic geodesics: vertical and semicircles perpendicular to boundary.
for x0 in [-1.6, 0.0, 1.6]:
    ax.plot([x0, x0], [0.05, 3.4], color=COLORS["blue"], lw=2.0, alpha=0.85)
for center, radius in [(-0.8, 0.85), (0.7, 0.9), (0.0, 1.6)]:
    theta = np.linspace(0.03, math.pi - 0.03, 240)
    ax.plot(center + radius * np.cos(theta), radius * np.sin(theta), color=COLORS["blue"], lw=1.4, alpha=0.65)

# Horocycles centered at infinity.
for y0 in [0.55, 1.05, 1.75, 2.55]:
    ax.plot(np.linspace(-2.5, 2.5, 250), np.full(250, y0), color=COLORS["green"], lw=1.5, alpha=0.8)

# Equidistant curves from the vertical geodesic x=0.
y = np.linspace(0.08, 3.3, 240)
equidistant_rows = []
for k in [-1.25, -0.65, 0.65, 1.25]:
    x_curve = k * y
    ax.plot(x_curve, y, color=COLORS["orange"], lw=1.8, alpha=0.82)
    distances = np.arcsinh(np.abs(x_curve) / y)
    equidistant_rows.append({"slope_k": k, "distance_value": float(distances.mean()), "distance_std": float(distances.std())})

ax.set_xlim(-3.0, 3.0)
ax.set_ylim(-0.05, 3.55)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Half-plane pencils: geodesics, horocycles, and equidistant curves")
ax.set_xlabel("x")
ax.set_ylabel("height")
ax.grid(alpha=0.25)
handles = [
    plt.Line2D([0], [0], color=COLORS["blue"], lw=2, label="hyperbolic lines"),
    plt.Line2D([0], [0], color=COLORS["green"], lw=2, label="horocycles"),
    plt.Line2D([0], [0], color=COLORS["orange"], lw=2, label="equidistant curves"),
]
ax.legend(handles=handles, loc="upper right")

pencils_path = save_png(fig, "half-plane-pencils-horocycles-equidistant.png")
pencils_table = write_csv(TABLES / "equidistant-curve-checks.csv", equidistant_rows)
table_artifacts.append(pencils_table)
computed_checks["pencils"] = {
    "equidistant_curve_count": len(equidistant_rows),
    "max_distance_std": max(row["distance_std"] for row in equidistant_rows),
    "horocycle_count": 4,
}
assert computed_checks["pencils"]["max_distance_std"] < 1e-12
display_many([pencils_path])


## 6. The Horosphere And The Euclidean Plane

The chapter ends by lifting the half-plane picture into three dimensions. In the upper-half-space model, horizontal planes are horospheres centered at the point at infinity. A remarkable theorem says that the geometry induced on a horosphere is Euclidean. In the model this is easy to inspect: horizontal translations slide a horosphere along itself, and equal Euclidean displacements on the same horizontal plane represent equal horospherical displacements.

The Plotly view is intentionally simple: vertical lines represent a bundle of geodesics with the same ideal endpoint, and horizontal planes represent horospheres. Rotate the scene and notice that the horosphere is not a small sphere in the drawing; it is the limiting surface of a family with center at infinity.

In [ ]:
grid = np.linspace(-1.6, 1.6, 18)
X, Y = np.meshgrid(grid, grid)
Z1 = np.full_like(X, 1.0)
Z2 = np.full_like(X, 2.0)
fig3 = go.Figure()
fig3.add_trace(go.Surface(x=X, y=Y, z=Z1, opacity=0.52, colorscale=[[0, "#bfdbfe"], [1, "#bfdbfe"]], showscale=False, name="horosphere z=1"))
fig3.add_trace(go.Surface(x=X, y=Y, z=Z2, opacity=0.26, colorscale=[[0, "#bbf7d0"], [1, "#bbf7d0"]], showscale=False, name="horosphere z=2"))
for x0 in [-1.2, -0.4, 0.4, 1.2]:
    for y0 in [-1.2, 0.0, 1.2]:
        fig3.add_trace(go.Scatter3d(x=[x0, x0], y=[y0, y0], z=[0.2, 3.0], mode="lines", line=dict(color="#111827", width=4), showlegend=False))
square = np.array([[-0.8, -0.8, 1.0], [0.8, -0.8, 1.0], [0.8, 0.8, 1.0], [-0.8, 0.8, 1.0], [-0.8, -0.8, 1.0]])
fig3.add_trace(go.Scatter3d(x=square[:, 0], y=square[:, 1], z=square[:, 2], mode="lines+markers", line=dict(color="#dc2626", width=6), marker=dict(size=4), name="Euclidean square on a horosphere"))
fig3.update_layout(
    title="Upper-half-space horospheres: horizontal Euclidean geometry",
    width=850,
    height=640,
    template="plotly_white",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="height", aspectmode="data"),
)
horosphere_path = HTML / "horosphere-euclidean-limit.html"
if horosphere_path.exists():
    horosphere_path.unlink()
fig3.write_html(horosphere_path, include_plotlyjs="cdn", full_html=True)
visual_artifacts.append(horosphere_path)

side_lengths = np.linalg.norm(np.diff(square, axis=0), axis=1)
computed_checks["horosphere"] = {
    "square_side_lengths": [float(v) for v in side_lengths],
    "side_length_std": float(side_lengths.std()),
    "vertical_geodesic_count": 12,
}
assert side_lengths.std() < 1e-12
display_many([horosphere_path])


## Reading The Chapter As A Model-Building Argument

The notebook's figures should be read as a consistency story. The hyperbolic axiom is not made plausible by a single dramatic picture; it becomes plausible because several models preserve the same structure while emphasizing different features. The half-plane shows multiple nonintersecting geodesics through a point. The disk models show why conformal and projective representations can both be legitimate. The angle-of-parallelism plot turns limiting parallels into a monotone function of distance. The area-defect diagram shows why hyperbolic triangles are finite even when asymptotic. The pencil diagram separates circles, horocycles, and equidistant curves. The horosphere view explains how a Euclidean plane can arise inside hyperbolic space as a limiting surface.

The computational checks are deliberately modest. They assert intersections, monotonicity, model-map bounds, additive identities, constant-distance formulas, and artifact integrity. Those are exactly the checks that keep the visual claims honest while leaving formal axiomatic proof to the surrounding course.

In [ ]:
manifest_rows = []
for artifact in visual_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "visual", "concept": artifact.stem.replace("-", " ")})
for artifact in table_artifacts:
    manifest_rows.append({"artifact": rel(artifact), "role": "table", "concept": artifact.stem.replace("-", " ")})
manifest_path = write_csv(TABLES / "artifact-manifest.csv", manifest_rows)
table_artifacts.append(manifest_path)

summary_payload = {
    "chapter": CHAPTER_NO,
    "title": CHAPTER["title"],
    "source_printed_pages": CHAPTER["printed"],
    "source_pdf_pages": CHAPTER["pdf"],
    "visuals": [rel(path) for path in visual_artifacts],
    "tables": [rel(path) for path in table_artifacts],
    "checks": computed_checks,
}
summary_path = write_json(CHECKS / "visual-summary.json", summary_payload)
legacy_summary_path = write_json(CHECKS / "visual_summary.json", summary_payload)
check_artifacts.extend([summary_path, legacy_summary_path])

final_sanity = {
    "visual_count": len(visual_artifacts),
    "table_count": len(table_artifacts),
    "parallel_nonmeeting_count": computed_checks["parallel_axiom"]["nonmeeting_count"],
    "angle_monotone": computed_checks["angle_of_parallelism"]["strictly_decreasing"],
    "all_defects_positive": computed_checks["area_defect"]["all_defects_positive"],
    "equidistant_std": computed_checks["pencils"]["max_distance_std"],
    "horosphere_side_std": computed_checks["horosphere"]["side_length_std"],
}
final_sanity_path = write_json(CHECKS / "final-sanity-summary.json", final_sanity)
check_artifacts.append(final_sanity_path)

assert_artifacts(visual_artifacts + table_artifacts + check_artifacts, min_bytes=80)
assert final_sanity["visual_count"] >= 6
assert final_sanity["parallel_nonmeeting_count"] > 2
assert final_sanity["angle_monotone"]
assert final_sanity["all_defects_positive"]
assert final_sanity["equidistant_std"] < 1e-12
assert final_sanity["horosphere_side_std"] < 1e-12
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()
assert not (TABLES / "artifact_manifest.csv").exists()

summary_payload


## Takeaways

- Hyperbolic geometry changes the parallel axiom while retaining enough absolute geometry to make incidence, order, and congruence meaningful.
- The Poincare models are conformal: they preserve angles while drawing hyperbolic lines as boundary-orthogonal arcs or vertical rays.
- The Klein model is projective: geodesics become straight chords, but angle measurement is no longer visually literal.
- The angle of parallelism `Pi(x) = 2 arctan(exp(-x))` decreases with distance, turning limiting parallels into a measurable hyperbolic phenomenon.
- Hyperbolic area is angular defect in the normalized unit: `area = pi - A - B - C`.
- Circles, horocycles, and equidistant curves are different trajectory families; the half-plane model makes their shapes easy to compare.
- A horosphere carries Euclidean geometry, which is one reason hyperbolic space can contain very familiar-looking limiting surfaces.